In [113]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, classification_report,
    precision_recall_curve, f1_score
)
from xgboost import XGBClassifier

##Load Data

In [114]:
main_df = pd.read_csv("patient_churn_main.csv")
val_df  = pd.read_csv("patient_churn_validation.csv")

print(f"Main shape : {main_df.shape}")
print(f"Val  shape : {val_df.shape}")

Main shape : (2000, 21)
Val  shape : (500, 11)


In [115]:
main_df.columns

Index(['PatientID', 'Age', 'Gender', 'State', 'Tenure_Months', 'Specialty',
       'Insurance_Type', 'Visits_Last_Year', 'Missed_Appointments',
       'Days_Since_Last_Visit', 'Last_Interaction_Date',
       'Overall_Satisfaction', 'Wait_Time_Satisfaction', 'Staff_Satisfaction',
       'Provider_Rating', 'Avg_Out_Of_Pocket_Cost', 'Billing_Issues',
       'Portal_Usage', 'Referrals_Made', 'Distance_To_Facility_Miles',
       'Churned'],
      dtype='object')

In [116]:
print("Shape:", main_df.shape)
print("\nColumns:\n", main_df.columns)

print("\nInfo:")
main_df.info()

print("\nMissing values:\n", main_df.isnull().sum())

Shape: (2000, 21)

Columns:
 Index(['PatientID', 'Age', 'Gender', 'State', 'Tenure_Months', 'Specialty',
       'Insurance_Type', 'Visits_Last_Year', 'Missed_Appointments',
       'Days_Since_Last_Visit', 'Last_Interaction_Date',
       'Overall_Satisfaction', 'Wait_Time_Satisfaction', 'Staff_Satisfaction',
       'Provider_Rating', 'Avg_Out_Of_Pocket_Cost', 'Billing_Issues',
       'Portal_Usage', 'Referrals_Made', 'Distance_To_Facility_Miles',
       'Churned'],
      dtype='object')

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   PatientID                   2000 non-null   object 
 1   Age                         2000 non-null   int64  
 2   Gender                      2000 non-null   object 
 3   State                       2000 non-null   object 
 4   Tenure_Months               2000 non-null   int6

In [117]:
for col in main_df.columns:
    print(col, "->", main_df[col].nunique())

PatientID -> 2000
Age -> 72
Gender -> 2
State -> 10
Tenure_Months -> 120
Specialty -> 7
Insurance_Type -> 4
Visits_Last_Year -> 16
Missed_Appointments -> 6
Days_Since_Last_Visit -> 679
Last_Interaction_Date -> 679
Overall_Satisfaction -> 36
Wait_Time_Satisfaction -> 36
Staff_Satisfaction -> 31
Provider_Rating -> 26
Avg_Out_Of_Pocket_Cost -> 1228
Billing_Issues -> 2
Portal_Usage -> 2
Referrals_Made -> 4
Distance_To_Facility_Miles -> 488
Churned -> 2


In [118]:
main_df['Churned'].value_counts()

,count
Churned,
1,1367
0,633


In [119]:
df = main_df.copy()

# Drop unnecessary columns
df = df.drop(['PatientID', 'Last_Interaction_Date'], axis=1)

# Feature Engineering
# Engagement score: actual visits minus missed ones
df["Engagement_Score"] = df["Visits_Last_Year"] - df["Missed_Appointments"]

# Cost efficiency
df["Cost_Per_Visit"] = df["Avg_Out_Of_Pocket_Cost"] / (df["Visits_Last_Year"] + 1)

# Composite satisfaction
df["Satisfaction_Avg"] = (
    df["Overall_Satisfaction"] +
    df["Wait_Time_Satisfaction"] +
    df["Staff_Satisfaction"]
) / 3

# Encode categoricals
df = pd.get_dummies(df, drop_first=True)

X_full = df.drop("Churned", axis=1)
y_full = df["Churned"]

In [120]:
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)

In [121]:
# Scaler — fit once on training set only
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

###Logistic Regression

In [122]:
# --- Logistic Regression ---
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_sc, y_train)
y_prob_lr = lr.predict_proba(X_test_sc)[:, 1]

###Random Forest Classifier

In [123]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=None,
    class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

### XGBoost

In [124]:
# --- XGBoost ---
xgb = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1, reg_lambda=1,
    eval_metric="logloss", random_state=42,
    use_label_encoder=False
)
xgb.fit(X_train, y_train)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

##Comparing Models

In [125]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "ROC-AUC": [
        roc_auc_score(y_test, y_prob_lr),
        roc_auc_score(y_test, y_prob_rf),
        roc_auc_score(y_test, y_prob_xgb),
    ]
}).sort_values("ROC-AUC", ascending=False)

print("\n── Full Model Comparison (ROC-AUC) ──")
print(comparison.to_string(index=False))


── Full Model Comparison (ROC-AUC) ──
              Model  ROC-AUC
      Random Forest 0.646664
            XGBoost 0.631825
Logistic Regression 0.614116


Threshold Tuning

In [126]:
precision, recall, thresholds = precision_recall_curve(y_test, y_prob_xgb)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_idx   = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"\nXGBoost best threshold (max-F1): {best_threshold:.3f}")
print(f"F1 at best threshold           : {f1_scores[best_idx]:.3f}")

y_pred_tuned = (y_prob_xgb >= best_threshold).astype(int)
print("\n── XGBoost Classification Report (tuned threshold) ──")
print(classification_report(y_test, y_pred_tuned))


XGBoost best threshold (max-F1): 0.226
F1 at best threshold           : 0.811

── XGBoost Classification Report (tuned threshold) ──
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       127
           1       0.68      1.00      0.81       273

    accuracy                           0.68       400
   macro avg       0.34      0.50      0.41       400
weighted avg       0.47      0.68      0.55       400



In [127]:
importances = pd.Series(rf.feature_importances_, index=X_full.columns)
top_features = importances.nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
top_features.plot(kind="barh", ax=ax, color="steelblue")
ax.set_title("Top 10 Features Influencing Patient Churn (Random Forest)")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance_full.png", dpi=150)
plt.close()
print("\nFeature importance chart saved → feature_importance_full.png")

print("\nTop 10 features:")
print(top_features.iloc[::-1].to_string())


Feature importance chart saved → feature_importance_full.png

Top 10 features:
Days_Since_Last_Visit         0.077066
Overall_Satisfaction          0.073033
Distance_To_Facility_Miles    0.070786
Avg_Out_Of_Pocket_Cost        0.068803
Tenure_Months                 0.066915
Age                           0.066624
Satisfaction_Avg              0.065870
Cost_Per_Visit                0.062632
Provider_Rating               0.055747
Wait_Time_Satisfaction        0.054716


## SHARED-FEATURE MODEL (for proper validation)

In [128]:
SHARED_RAW = ["Age", "Gender", "Tenure_Months", "Insurance_Type",
              "Visits_Last_Year", "Missed_Appointments"]

def engineer_shared(df_in, target_col):
    """Feature engineering that works on shared columns only."""
    d = df_in[SHARED_RAW + [target_col]].copy()
    d["Engagement_Score"] = d["Visits_Last_Year"] - d["Missed_Appointments"]
    d["Missed_Rate"] = d["Missed_Appointments"] / (d["Visits_Last_Year"] + 1)
    # Fill NaNs in Insurance_Type before encoding
    d["Insurance_Type"] = d["Insurance_Type"].fillna("Unknown")
    d = pd.get_dummies(d, drop_first=True)
    return d

train_shared = engineer_shared(main_df, "Churned")
X_sh = train_shared.drop("Churned", axis=1)
y_sh = train_shared["Churned"]

X_sh_train, X_sh_test, y_sh_train, y_sh_test = train_test_split(
    X_sh, y_sh, test_size=0.2, random_state=42, stratify=y_sh
)

xgb_shared = XGBClassifier(
    n_estimators=300, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42,
    use_label_encoder=False
)
xgb_shared.fit(X_sh_train, y_sh_train)
y_prob_sh = xgb_shared.predict_proba(X_sh_test)[:, 1]

print(f"\n── Shared-Feature Model ROC-AUC (held-out test) ──")
print(f"  {roc_auc_score(y_sh_test, y_prob_sh):.4f}")


── Shared-Feature Model ROC-AUC (held-out test) ──
  0.5329


## Validate on Validation CSV

In [129]:
val_shared = engineer_shared(val_df.rename(columns={"Churn": "Churn"}), "Churn")
# Note: validation target column is "Churn", not "Churned"

X_val = val_shared.drop("Churn", axis=1)
y_val = val_shared["Churn"]

# Align columns to training feature set
X_val = X_val.reindex(columns=X_sh.columns, fill_value=0)

y_prob_val  = xgb_shared.predict_proba(X_val)[:, 1]
y_pred_val  = (y_prob_val >= 0.5).astype(int)

print("\n── Validation Set Results (shared-feature XGBoost) ──")
print(f"  ROC-AUC : {roc_auc_score(y_val, y_prob_val):.4f}")
print(classification_report(y_val, y_pred_val))

print("\nDone ✅  All outputs saved.")


── Validation Set Results (shared-feature XGBoost) ──
  ROC-AUC : 0.5131
              precision    recall  f1-score   support

           0       1.00      0.01      0.01       376
           1       0.25      1.00      0.40       124

    accuracy                           0.25       500
   macro avg       0.62      0.50      0.20       500
weighted avg       0.81      0.25      0.11       500


Done ✅  All outputs saved.


In [130]:
# ── 6. SAVE MODEL FILES (required by app.py) ──────────────────────────────────
import joblib, os

# Use RF for the app — best ROC-AUC and better-calibrated threshold than XGBoost
# Tune threshold on RF (same logic as XGBoost above)
from sklearn.metrics import precision_recall_curve

precision_rf, recall_rf, thresholds_rf = precision_recall_curve(y_test, y_prob_rf)
f1_rf = 2 * precision_rf * recall_rf / (precision_rf + recall_rf + 1e-9)

# Pick best threshold where precision >= 0.5 (avoids the "predict everyone" collapse)
best_thresh = 0.5
best_f1_val = 0
for t, p, r, f in zip(thresholds_rf, precision_rf[:-1], recall_rf[:-1], f1_rf[:-1]):
    if p >= 0.5 and f > best_f1_val:
        best_f1_val = f
        best_thresh = t

print(f"\nRF best threshold (precision ≥ 0.5): {best_thresh:.3f}  |  F1: {best_f1_val:.3f}")

os.makedirs("model", exist_ok=True)
joblib.dump(rf,           "model/churn_model.pkl")
joblib.dump(list(X_full.columns), "model/model_columns.pkl")
joblib.dump(best_thresh,  "model/best_threshold.pkl")

print("\n✅ Saved:")
print("   model/churn_model.pkl")
print("   model/model_columns.pkl")
print("   model/best_threshold.pkl")
print("\nCopy the model/ folder next to app.py and run:  streamlit run app.py")


RF best threshold (precision ≥ 0.5): 0.457  |  F1: 0.812

✅ Saved:
   model/churn_model.pkl
   model/model_columns.pkl
   model/best_threshold.pkl

Copy the model/ folder next to app.py and run:  streamlit run app.py
